# jG_00 - Audit SFM terdistribusi: hanya ringkasan statistik yang meninggalkan klien (JURNAL)
**Tujuan:** (1) tiap klien hanya mengirim median dan p99 per fitur; (2) server memberi label tiap pasangan dengan aturan SFM paper1
dan menetapkan tingkat fitur (inti vs ekstensi); (3) menguji kestabilan label terhadap derau pada ringkasan; (4) menghitung biaya komunikasi.
**Aturan label (paper1, Pers. 2-3):** rasio terkoreksi-satuan `r' = c x ratio` untuk median dan p99; `aligned` bila
`max(|log10 r'_med|, |log10 r'_p99|) <= 1`, selain itu `scale-mismatch`; fitur laju tanpa `c` konstan = `rate-proxy`.
**Catatan jujur:** keputusan memasukkan fitur di paper1 terutama semantik (dokumentasi alat + faktor satuan `c`); rasio hanya memberi label.
Uji derau di sini BUKAN jaminan privasi diferensial formal.
**Kriteria H3 (disepakati sebelum melihat angka):** label 7 pasangan satuan-eksak (cic2018 vs unsw) sama dengan Tabel 2 paper1
(duration scale, spkts aligned, dpkts scale, sbytes scale, dbytes scale, smean aligned, dmean aligned). Selisih dilaporkan apa adanya.
**Output:** `journal_out_fsfm/audit_*.csv|json`, S3 `federated_journal/fsfm/audit/`. Uji kering: `FSFM_MOCK=1 FSFM_FAST=1`.

In [ ]:
import importlib, sys, subprocess, os
MOCK = os.environ.get('FSFM_MOCK', '0') == '1'   # uji kering dengan data sintetik (tanpa S3)
FAST = os.environ.get('FSFM_FAST', '0') == '1'   # konfigurasi kecil untuk uji cepat
need = [('numpy','numpy'),('pandas','pandas'),('sklearn','scikit-learn'),('scipy','scipy')] + ([] if MOCK else [('boto3','boto3')])
for mod, pkg in need:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg], check=True)
import io, json, time, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, roc_auc_score

# ---------------- KONFIG ----------------
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
FAR_ROOT='unsw-far/federated_survey'            # klien: cic, unsw (9 fitur selaras)
NEAR_ROOT='federated_journal/neardomain'        # klien: cic2017, cic2018 (76 fitur)
OUT='federated_journal/fsfm'
OUTDIR='journal_out_fsfm'; os.makedirs(OUTDIR, exist_ok=True)
CLIENTS=['cic2017','cic2018','unsw']
SEEDS=[42,13,101,202,303]
ROUNDS=30; LOCAL_EPOCHS=1; LR=1e-3; BATCH=256; HID=[128,64]; MU=0.01
MAX_TRAIN=100000; LOCAL_ONLY_EPOCHS=15; CENTRAL_EPOCHS=10
if FAST:
    SEEDS=[42,13]; ROUNDS=3; MAX_TRAIN=3000; LOCAL_ONLY_EPOCHS=2; CENTRAL_EPOCHS=2
print('MOCK =',MOCK,'| FAST =',FAST,'| seeds =',SEEDS,'| rounds =',ROUNDS)


In [ ]:
NINE=['flow duration','total fwd packets','total backward packets',
      'total length of fwd packets','total length of bwd packets',
      'fwd packet length mean','bwd packet length mean','flow bytes/s','bwd packets/s']
ALIGN_KEYS=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
D_CORE=9
if not MOCK:
    import boto3
    s3=boto3.client('s3',region_name=REGION)
    def npz(key): return np.load(io.BytesIO(s3.get_object(Bucket=BUCKET,Key=key)['Body'].read()),allow_pickle=True)
    def load_raw(root,c):
        D={}
        for sp in ('train','test'):
            d=npz(f'{root}/data/{c}/{c}_{sp}.npz'); D[sp]=(np.asarray(d['X'],float),np.asarray(d['y'],int)); D['feats']=[str(f) for f in d['feats']]
        return D

def mock_client(n,d,seed,shift):
    rng=np.random.RandomState(seed); Z=rng.randn(n,d)
    y=(Z[:,:4].sum(1)+0.7*rng.randn(n)>0.5).astype(int)
    Z[y==1,:6]+=0.8
    if d>15: Z[y==1,9:15]+=0.8
    return np.expm1(np.abs(Z+shift)*1.5), y

DATA={}
rs0=np.random.RandomState(0)
def subsample(X,y):
    if MAX_TRAIN and len(y)>MAX_TRAIN:
        i=rs0.choice(len(y),MAX_TRAIN,replace=False); return X[i],y[i]
    return X,y
if MOCK:
    names76=NINE+[f'ext_{i}' for i in range(67)]
    for k,c in enumerate(('cic2017','cic2018')):
        Xtr,ytr=mock_client(4000,76,10+k,0.1*k); Xte,yte=mock_client(1500,76,20+k,0.1*k)
        DATA[c]=dict(train=(Xtr,ytr),test=(Xte,yte),feats=names76)
    Xtr,ytr=mock_client(4000,9,30,0.3); Xte,yte=mock_client(1500,9,31,0.3)
    DATA['unsw']=dict(train=(Xtr,ytr),test=(Xte,yte),feats=NINE)
else:
    for c in ('cic2017','cic2018'):
        D=load_raw(NEAR_ROOT,c); f=D['feats']
        core=[f.index(n) for n in NINE]; ext=[i for i in range(len(f)) if i not in core]; order=core+ext
        DATA[c]=dict(train=subsample(D['train'][0][:,order],D['train'][1]),
                     test=(D['test'][0][:,order],D['test'][1]),feats=[f[i] for i in order])
    U=load_raw(FAR_ROOT,'unsw')
    assert U['feats']==ALIGN_KEYS, f'urutan fitur UNSW tidak sesuai ALIGN: {U["feats"]}'
    DATA['unsw']=dict(train=subsample(*U['train']),test=U['test'],feats=NINE)
D_EXT=len(DATA['cic2017']['feats'])-D_CORE
assert len(DATA['cic2018']['feats'])-D_CORE==D_EXT and DATA['cic2017']['feats']==DATA['cic2018']['feats']
for c in CLIENTS:
    X,y=DATA[c]['train']; print(c,'train',X.shape,'pos=%.3f'%y.mean(),'| test',DATA[c]['test'][0].shape,'| max kolom 0..2 =',np.round(X[:,:3].max(0),1))
print('D_CORE =',D_CORE,'| D_EXT =',D_EXT)


In [ ]:
# ringkasan yang dikirim klien (satu-satunya yang keluar): median dan p99 per fitur dari data TRAIN
def summarize(X): return np.median(X,0), np.percentile(X,99,axis=0)
S={c:summarize(DATA[c]['train'][0]) for c in CLIENTS}
SPEC=[('duration',1e-6,'exact'),('fwd_pkts',1,'exact'),('bwd_pkts',1,'exact'),('fwd_bytes',1,'exact'),
      ('bwd_bytes',1,'exact'),('fwd_mean',1,'exact'),('bwd_mean',1,'exact'),('src_load',8,'proxy-dir'),('dst_load',None,'proxy-rate')]
PAPER1={'duration':'scale-mismatch','fwd_pkts':'aligned','bwd_pkts':'scale-mismatch','fwd_bytes':'scale-mismatch',
        'bwd_bytes':'scale-mismatch','fwd_mean':'aligned','bwd_mean':'aligned','src_load':'rate-proxy','dst_load':'rate-proxy'}
def ratio(a,b):
    if a==0 and b==0: return 1.0
    if a==0 or b==0: return np.inf
    return a/b
def verdict(ma,pa,mb,pb,c,kind):
    if kind!='exact' or c is None: return 'rate-proxy'
    r1=c*ratio(ma,mb); r2=c*ratio(pa,pb)
    if (not np.isfinite(r1)) or (not np.isfinite(r2)) or r1<=0 or r2<=0: return 'scale-mismatch'
    return 'aligned' if max(abs(np.log10(r1)),abs(np.log10(r2)))<=1 else 'scale-mismatch'

# --- bagian 1: cic vs unsw pada 9 fitur inti ---
rows=[]
for cic in ('cic2017','cic2018'):
    for k,(name,c,kind) in enumerate(SPEC):
        ma,pa=S[cic][0][k],S[cic][1][k]; mb,pb=S['unsw'][0][k],S['unsw'][1][k]
        rows.append(dict(pair=f'{cic} vs unsw',feature=name,med_a=ma,med_b=mb,p99_a=pa,p99_b=pb,c=c,verdict=verdict(ma,pa,mb,pb,c,kind),paper1=PAPER1[name]))
A1=pd.DataFrame(rows); print(A1.to_string(index=False,float_format=lambda v:f'{v:.4g}'))
for cic in ('cic2017','cic2018'):
    g=A1[(A1.pair==f'{cic} vs unsw')&(A1.paper1!='rate-proxy')]; ok=int((g.verdict==g.paper1).sum())
    print(f'{cic}: label sama dengan paper1 pada {ok} dari {len(g)} pasangan satuan-eksak -> H3 {"LOLOS" if ok==len(g) else "TIDAK SEMUA SAMA"} (untuk pasangan ini)')

# --- bagian 2: cic2017 vs cic2018 pada seluruh fitur (alat sama, c=1) ---
F=DATA['cic2017']['feats']; rows2=[]
for k,name in enumerate(F):
    ma,pa=S['cic2017'][0][k],S['cic2017'][1][k]; mb,pb=S['cic2018'][0][k],S['cic2018'][1][k]
    rows2.append(dict(feature=name,tier='core' if k<D_CORE else 'ext',verdict=verdict(ma,pa,mb,pb,1,'exact')))
A2=pd.DataFrame(rows2); print('\ncic2017 vs cic2018 (semua fitur):'); print(A2.verdict.value_counts().to_string())
print('Pasangan scale-mismatch tetap dipakai (kebijakan paper1: skala diredam z-score per klien); hanya different-feature yang ditolak.')

In [ ]:
# --- bagian 3: kestabilan label terhadap derau multiplikatif pada ringkasan ---
rng=np.random.RandomState(0); TR=40 if FAST else 200
def labels_with_noise(sig):
    def nz(v): return v*np.exp(rng.randn(*np.shape(v))*sig) if sig>0 else v
    N={c:(nz(S[c][0]),nz(S[c][1])) for c in CLIENTS}
    l1=[verdict(N['cic2018'][0][k],N['cic2018'][1][k],N['unsw'][0][k],N['unsw'][1][k],c,kind) for k,(n,c,kind) in enumerate(SPEC)]
    l2=[verdict(N['cic2017'][0][k],N['cic2017'][1][k],N['cic2018'][0][k],N['cic2018'][1][k],1,'exact') for k in range(len(F))]
    return l1,l2
b1,b2=labels_with_noise(0.0); rob=[]
for sig in (0.0,0.05,0.1,0.2,0.5,1.0):
    s1=[];s2=[]
    for _ in range(TR if sig>0 else 1):
        l1,l2=labels_with_noise(sig); s1.append(np.mean([a==b for a,b in zip(l1,b1)])); s2.append(np.mean([a==b for a,b in zip(l2,b2)]))
    rob.append(dict(sigma=sig,stab_cic18_vs_unsw=round(float(np.mean(s1)),3),stab_cic17_vs_cic18=round(float(np.mean(s2)),3)))
R=pd.DataFrame(rob); print('Kestabilan label (fraksi pasangan yang labelnya tidak berubah), derau lognormal sigma pada median/p99:'); print(R.to_string(index=False))

# --- bagian 4: biaya komunikasi ---
comm=[]
for c in CLIENTS:
    nf=DATA[c]['train'][0].shape[1]; b_sum=nf*2*8; b_raw=DATA[c]['train'][0].nbytes
    comm.append(dict(client=c,features=nf,summary_bytes=b_sum,raw_train_bytes=b_raw,ratio=round(b_sum/b_raw,8)))
C=pd.DataFrame(comm); print('\nBiaya komunikasi audit (ringkasan vs data mentah):'); print(C.to_string(index=False))

# --- bagian 5: tabel tingkat fitur ---
T=pd.DataFrame(dict(feature=F,tier=['core']*D_CORE+['ext']*D_EXT,owners=['cic2017,cic2018,unsw']*D_CORE+['cic2017,cic2018']*D_EXT))
for f_,df_ in (('audit_cic_vs_unsw.csv',A1),('audit_cic17_vs_cic18.csv',A2),('audit_robustness.csv',R),('audit_comm.csv',C),('audit_tiers.csv',T)):
    df_.to_csv(os.path.join(OUTDIR,f_),index=False)
json.dump(dict(core=F[:D_CORE],ext_count=D_EXT,mock=MOCK,fast=FAST),open(os.path.join(OUTDIR,'audit_tiers.json'),'w'),indent=1)
if not MOCK:
    for f_ in ('audit_cic_vs_unsw.csv','audit_cic17_vs_cic18.csv','audit_robustness.csv','audit_comm.csv','audit_tiers.csv','audit_tiers.json'):
        s3.upload_file(os.path.join(OUTDIR,f_),BUCKET,f'{OUT}/audit/{f_}')
    print('upload ke s3://%s/%s/audit/'%(BUCKET,OUT))
print('Jangan menyimpulkan sebelum Kiro membaca hasilnya.')